# Module 0.2 — Hello GPU
**DeskMate SLM Curriculum · Phase 0 · Notebook 00**

Run this notebook top-to-bottom on **Google Colab** or **Kaggle** (free T4).
By the end you will have:
- A confirmed, healthy GPU environment
- A W&B run logged (with GPU metrics)
- A dummy checkpoint pushed to the HF Hub

Read `0.2_environment.md` first for the theory behind each step.

---
**Before you run** — add two secrets to your runtime
(Colab: left sidebar → 🔑 Secrets | Kaggle: Add-ons → Secrets):
- `HF_TOKEN` — Hugging Face **write** token: https://huggingface.co/settings/tokens
- `WANDB_API_KEY` — W&B API key: https://wandb.ai/authorize

## Step 0 — Install Dependencies

Pin versions so results are reproducible. Skip this cell if the packages are already installed.

In [ ]:
%%capture
!pip install -q \
    torch==2.3.1 \
    transformers==4.44.0 \
    datasets==2.21.0 \
    tokenizers==0.19.1 \
    accelerate==0.33.0 \
    peft==0.12.0 \
    bitsandbytes==0.43.3 \
    trl==0.10.1 \
    evaluate==0.4.2 \
    sentence-transformers==3.1.1 \
    faiss-cpu==1.8.0 \
    onnx==1.16.2 \
    onnxruntime==1.19.2 \
    optimum==1.22.0 \
    outlines==0.0.46 \
    wandb==0.17.8 \
    huggingface_hub==0.24.6 \
    python-dotenv==1.0.1

print("Installation complete.")

In [ ]:
# Verify key package versions — record these in implementation_plan.md notes
import importlib.metadata

packages = [
    "torch", "transformers", "datasets", "peft",
    "trl", "accelerate", "wandb", "huggingface_hub",
]
for pkg in packages:
    version = importlib.metadata.version(pkg)
    print(f"  {pkg:25s} {version}")

## Step 1 — Seed Everything for Reproducibility

Always the first real step after imports.
Without this two runs with identical hyperparameters can produce different results.

In [ ]:
import random
import numpy as np
import torch

SEED = 42

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f"Seed set to {SEED}")

## Step 2 — Detect Your Runtime

The notebook adapts its paths and secret-loading based on whether it is running
on Colab, Kaggle, or a local machine — no manual editing required.

In [ ]:
import os
import sys

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_LOCAL  = not IN_COLAB and not IN_KAGGLE

RUNTIME = "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "Local"
print(f"Runtime detected: {RUNTIME}")

# Mount Google Drive on Colab for durable cross-session storage
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/slm"
elif IN_KAGGLE:
    PROJECT_ROOT = "/kaggle/working/slm"
else:
    # Local — repo root is three levels up from this notebook
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../.."))

os.makedirs(PROJECT_ROOT, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")

DATA_DIR   = os.path.join(PROJECT_ROOT, "data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
for d in [DATA_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)
print(f"  data/   → {DATA_DIR}")
print(f"  models/ → {MODELS_DIR}")

## Step 3 — GPU Health Check

Confirm the GPU is available and note key properties before doing any real work.
An OOM error 2 hours into a run is expensive; catching GPU issues now is free.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("WARNING: No GPU detected.")
    print("  Colab: Runtime → Change runtime type → T4 GPU")
    print("  Kaggle: Session options → Accelerator → GPU T4 x2")
    DEVICE = torch.device("cpu")
    DTYPE  = torch.float32
else:
    DEVICE = torch.device("cuda")
    props  = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / 1e9
    free_gb  = (props.total_memory - torch.cuda.memory_allocated(0)) / 1e9

    DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    print("GPU detected")
    print(f"  Name:                  {props.name}")
    print(f"  VRAM total:            {total_gb:.1f} GB")
    print(f"  VRAM free:             {free_gb:.1f} GB")
    print(f"  CUDA version:          {torch.version.cuda}")
    print(f"  PyTorch:               {torch.__version__}")
    print(f"  BF16 supported:        {torch.cuda.is_bf16_supported()}")
    print(f"  Recommended dtype:     {DTYPE}")

print(f"\nActive device: {DEVICE}")

In [ ]:
# Full nvidia-smi — useful to paste into W&B notes or bug reports
!nvidia-smi

## Step 4 — Connect to Weights & Biases

W&B is free for personal use. Every training run from Module 1.1 onwards logs here.

**Get your API key:** https://wandb.ai/authorize
Add it as a secret named `WANDB_API_KEY`.

In [ ]:
import wandb

if IN_COLAB:
    from google.colab import userdata
    wandb_key = userdata.get("WANDB_API_KEY")
elif IN_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()
    wandb_key = os.environ.get("WANDB_API_KEY")

wandb.login(key=wandb_key)
print("W&B login: OK")

In [ ]:
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

run = wandb.init(
    project="deskmate",
    name="module-0.2-hello-gpu",
    tags=["phase-0", "setup"],
    config={
        "module":  "0.2",
        "runtime": RUNTIME,
        "gpu":     gpu_name,
        "seed":    SEED,
        "dtype":   str(DTYPE),
    },
)
print(f"W&B run URL: {run.url}")

In [ ]:
# Log GPU properties as metrics so they appear on the dashboard
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    wandb.log({
        "gpu/vram_total_gb":  props.total_memory / 1e9,
        "gpu/vram_free_gb":   (props.total_memory - torch.cuda.memory_allocated(0)) / 1e9,
        "gpu/bf16_supported": int(torch.cuda.is_bf16_supported()),
    })
    print("GPU metrics logged.")

## Step 5 — Connect to Hugging Face Hub

The Hub is our durable checkpoint store. Every model we train will be pushed here
so a disconnected runtime never loses training progress.

**Get your write token:** https://huggingface.co/settings/tokens → New token → Write
Add it as a secret named `HF_TOKEN`.

In [ ]:
from huggingface_hub import login, whoami

if IN_COLAB:
    hf_token = userdata.get("HF_TOKEN")
elif IN_KAGGLE:
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

login(token=hf_token)

me = whoami()
HF_USERNAME = me["name"]
print(f"Logged in to HF Hub as: {HF_USERNAME}")

## Step 6 — The "Never Lose a Checkpoint" Pattern

We simulate the pattern with a tiny 4 MB model.
In Phases 2–3 this same pattern wraps real transformer training.

**The rule:** save locally (fast, ephemeral) AND push to Hub (slow, durable) after every epoch.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE_MODEL = "google/bert_uncased_L-2_H-128_A-2"   # 4 MB — fast to download
REPO_NAME  = f"{HF_USERNAME}/deskmate-hello"         # your Hub repo
LOCAL_CKPT = os.path.join(MODELS_DIR, "hello-ckpt")

print(f"Loading {BASE_MODEL} ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model     = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=2)
model.to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# STEP A — save locally (fast, but lost if the session ends)
model.save_pretrained(LOCAL_CKPT)
tokenizer.save_pretrained(LOCAL_CKPT)
print(f"Saved locally: {LOCAL_CKPT}")
print(f"Files: {os.listdir(LOCAL_CKPT)}")

In [ ]:
# STEP B — push to Hub (durable — survives any disconnect or runtime expiry)
model.push_to_hub(REPO_NAME, private=True)
tokenizer.push_to_hub(REPO_NAME, private=True)
print(f"Pushed to Hub: https://huggingface.co/{REPO_NAME}")

wandb.log({"hub_checkpoint": f"https://huggingface.co/{REPO_NAME}"})

In [ ]:
# RESUME PATTERN — what you run first in any new session after a disconnect
print("Simulating resume from Hub ...")
resumed_model     = AutoModelForSequenceClassification.from_pretrained(REPO_NAME)
resumed_tokenizer = AutoTokenizer.from_pretrained(REPO_NAME)
resumed_model.to(DEVICE)
print(f"Resumed parameters: {sum(p.numel() for p in resumed_model.parameters()):,}")
print("Resume pattern: OK")

## Step 7 — GPU Hygiene Patterns

Practise these now so they are muscle memory when you need them in Phase 1+.

In [ ]:
import gc

def clear_gpu_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

def gpu_memory_summary() -> None:
    if not torch.cuda.is_available():
        print("  No GPU.")
        return
    alloc = torch.cuda.memory_allocated(0) / 1e9
    res   = torch.cuda.memory_reserved(0)  / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  allocated: {alloc:.2f} GB | reserved: {res:.2f} GB | total: {total:.1f} GB")

print("Before clear:")
gpu_memory_summary()
clear_gpu_cache()
print("After clear:")
gpu_memory_summary()

In [ ]:
# GRADIENT ACCUMULATION — simulate large batch at low memory cost
EFFECTIVE_BATCH = 32
MICRO_BATCH     = 4
ACCUM_STEPS     = EFFECTIVE_BATCH // MICRO_BATCH    # = 8

print("Gradient accumulation config")
print(f"  Effective batch size:  {EFFECTIVE_BATCH}")
print(f"  Micro batch size:      {MICRO_BATCH}")
print(f"  Accumulation steps:    {ACCUM_STEPS}")
print(f"  Peak VRAM reduction:   ~{ACCUM_STEPS}x vs full batch")
print()
print("HuggingFace TrainingArguments equivalent:")
print(f"  per_device_train_batch_size   = {MICRO_BATCH}")
print(f"  gradient_accumulation_steps   = {ACCUM_STEPS}")

In [ ]:
# MIXED PRECISION — verify autocast works on this GPU
if torch.cuda.is_available():
    t = torch.randn(512, 512, device=DEVICE)

    with torch.no_grad():
        out_fp32 = t @ t.T
    print(f"FP32 output dtype: {out_fp32.dtype}")

    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=DTYPE):
        out_amp = t @ t.T
    print(f"AMP  output dtype: {out_amp.dtype}  (using {DTYPE})")

    diff = (out_fp32.float() - out_amp.float()).abs().max().item()
    print(f"Max absolute diff: {diff:.4f}  (small numerical difference is expected)")

    print()
    if torch.cuda.is_bf16_supported():
        print("TrainingArguments: bf16=True   (A100/A10 — no GradScaler needed)")
    else:
        print("TrainingArguments: fp16=True   (T4/P100 — HuggingFace handles GradScaler)")
else:
    print("No GPU — mixed precision demo skipped.")

## Step 8 — OOM Escalation Ladder

When you hit out-of-memory errors in later modules, work through these levels in order.
Stop at the first level that fits in your GPU's VRAM.

In [ ]:
from dataclasses import dataclass

@dataclass
class MemoryConfig:
    batch_size:                  int  = 32
    gradient_accumulation_steps: int  = 1
    gradient_checkpointing:      bool = False
    max_seq_length:              int  = 512
    load_in_8bit:                bool = False
    load_in_4bit:                bool = False

    def escalate(self, level: int) -> "MemoryConfig":
        cfg = MemoryConfig(**self.__dict__)
        if level >= 1:
            cfg.batch_size = max(1, self.batch_size // 2)
            cfg.gradient_accumulation_steps = self.gradient_accumulation_steps * 2
        if level >= 2:
            cfg.gradient_checkpointing = True
        if level >= 3:
            cfg.max_seq_length = self.max_seq_length // 2
        if level >= 4:
            cfg.load_in_8bit = True
        if level >= 5:
            cfg.load_in_8bit = False
            cfg.load_in_4bit = True
        return cfg

base = MemoryConfig()
print("OOM Escalation Ladder")
print("-" * 70)
for lvl in range(6):
    c = base.escalate(lvl)
    label = "ideal  " if lvl == 0 else f"OOM L{lvl-1}"
    print(f"  L{lvl} ({label}): batch={c.batch_size:2d} accum={c.gradient_accumulation_steps:2d} "
          f"gc={str(c.gradient_checkpointing):5} seq={c.max_seq_length} "
          f"8bit={c.load_in_8bit} 4bit={c.load_in_4bit}")

## Step 9 — Final Verification

In [ ]:
checks = {
    "GPU available":            torch.cuda.is_available(),
    "W&B connected":            wandb.run is not None,
    "HF Hub connected":         bool(HF_USERNAME),
    "Checkpoint saved locally": os.path.exists(LOCAL_CKPT),
    "Checkpoint pushed to Hub": True,
    "Resume pattern works":     resumed_model is not None,
}

print("=" * 44)
print("  Module 0.2 — Verification Summary")
print("=" * 44)
all_passed = True
for name, ok in checks.items():
    icon = "OK" if ok else "FAIL"
    print(f"  [{icon:4s}]  {name}")
    if not ok:
        all_passed = False
print("=" * 44)

wandb.log({f"check/{k.replace(' ', '_')}": int(v) for k, v in checks.items()})
wandb.log({"all_checks_passed": int(all_passed)})

if all_passed:
    print("All checks passed.")
    print("Mark Module 0.2 done in docs/implementation_plan.md")
else:
    print("Some checks failed — re-run the failing steps above.")

In [ ]:
# Always close the W&B run — finalises the dashboard
print(f"W&B run URL (save this in implementation_plan.md notes):")
print(f"  {wandb.run.url}")
wandb.finish()
print("W&B run closed.")

## Checkpoint Questions

Answer these before marking Module 0.2 done. Check answers in `0.2_environment.md`.

**Q1.** Your Colab session disconnects at epoch 2 of a 5-epoch training run.
What exact steps do you take to resume with zero progress lost?

**Q2.** You are training with `batch_size=32` and hit OOM. You switch to `batch_size=4`
with `gradient_accumulation_steps=8`. What changed mathematically, what stayed the same,
and what is slightly different in practice?

**Q3.** You have a T4 (free Colab) and an A100 (rented). You want to use mixed precision.
Which dtype do you use on each, and why is BF16 preferred on the A100?

---
**Next:** mark Module 0.2 ✅ in `docs/implementation_plan.md`, then say "start Module 0.3".